# 02 — Validated Multi-Temporal SAR & Ground-Truth Training Data Pipeline

Acquires **strictly validated** multi-temporal SAR + topography inputs (**X**) and **cross-sensor validated ground-truth labels** (**y**) for training the 6-class SAR surface classification model (`0: Bare Ground`, `1: Water & Flood`, `2: Tree Lines`, `3: Wetlands`, `4: Cropland`, `5: Urban`).

### Why a Dedicated Training Pipeline (vs. `01_daten_suchen_und_laden.ipynb`)?
1. **Validated Science Products Only:** Queries `NISAR_L2_GCOV_V1`, `NISAR_L2_GUNW_V1`, `OPERA_L2_RTC-S1_V1`, `OPERA_L3_DSWX-HLS_V1`, and `NASADEM_HGT` (`v001`) — **no `PROVISIONAL` fallback** by default (`ALLOW_PROVISIONAL_NISAR = False`).
2. **Quality & Geometry Masking:** Streams `OPERA_L2_RTC-S1` `*_mask.tif` (excluding radar layover, shadow, and invalid pixels), NISAR `mask` / `numberOfLooks`, and masks steep terrain (`slope_deg > MAX_SLOPE_DEG`).
3. **Cross-Sensor Validated Labels (`y`):**
   - **Static Multi-Class Baseline:** **ESA WorldCover 10m v200 (2021)** streamed AOI-only via Cloud-Optimized GeoTIFF range requests and mapped to project classes `0..5`.
   - **Dynamic Water / Flood Validation:** **OPERA L3 `DSWx-HLS` (`OPERA_L3_DSWX-HLS_V1`)** (optical HLS-derived, independent from SAR!) filtered to cloud/snow-free high-confidence pixels and temporally matched (`±DSWX_MAX_TIME_DIFF_DAYS`) to update seasonal water and inundated vegetation labels.
   - **Patch Homogeneity Filter:** Optionally keeps only pixels whose `3x3` label neighborhood is uniform (`REQUIRE_HOMOGENEOUS_3X3 = True`), avoiding mixed-boundary label noise.
4. **Leakage-Free Spatial-Block Split & Stratified Export:** Splits each AOI on a `BLOCK_SIZE_M` spatial grid into disjoint `train` / `val` / `test` sets across all timestamps, then exports class-balanced `.npz`, `.parquet`, NetCDF-4 cubes, and a validation manifest under `data/training/<RUN_NAME>/`.


## 1. Setup, Dependencies & Earthdata Authentication

In [ ]:
import hashlib
import json
import math
import os
import zipfile
from collections import defaultdict
from pathlib import Path

import earthaccess
import geopandas as gpd
import h5py
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterio.features
import rioxarray  # noqa: F401 (registers .rio accessor)
import xarray as xr
from rasterio.enums import Resampling
from rioxarray.exceptions import NoDataInBounds
from scipy.ndimage import binary_dilation, maximum_filter, minimum_filter, uniform_filter
from shapely.geometry import box

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), Path.cwd().parent] if (p / ".env").exists()),
    Path.cwd(),
)
ENV_PATH = PROJECT_ROOT / ".env"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
TRAINING_DIR = PROJECT_ROOT / "data" / "training"
RAW_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_DIR.mkdir(parents=True, exist_ok=True)

if ENV_PATH.exists():
    for line in ENV_PATH.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip().strip("\"'"))

if not os.environ.get("EARTHDATA_TOKEN"):
    raise ValueError("EARTHDATA_TOKEN not found. Please set it in .env.")

auth = earthaccess.login(strategy="environment")
print("Earthdata authenticated:", auth.authenticated)
print("Raw cache dir:          ", RAW_DIR)
print("Training output dir:    ", TRAINING_DIR)


## 2. Configuration: Training AOIs, Strict Validation Rules & Class Schema

Configure `TRAINING_AOIS` (multi-biome regions improve generalization), temporal bounds, strict validation filters, and spatial-block splitting parameters.
* **Tip:** Start with 1 AOI (`"oberrhein"`) and `MAX_TIMESTEPS_PER_AOI = 6` for a fast end-to-end check, then enable additional AOIs (e.g., `"danube_delta"`, `"po_valley"`) for full training.


In [ ]:
# --- Run & AOI Configuration ---
RUN_NAME = "validated_train_v1"

# (west, south, east, north) in EPSG:4326
TRAINING_AOIS = {
    "oberrhein": (8.28, 49.02, 8.58, 49.28),       # Forest, cropland, river, urban
    # "danube_delta": (29.00, 44.95, 29.40, 45.25), # Extensive wetlands, water, reedbeds
    # "po_valley": (11.35, 44.80, 11.75, 45.10),    # Cropland, floodplain, urban
}

START_DATE = "2025-01-01"
END_DATE = "2026-11-14"
MAX_TIMESTEPS_PER_AOI = 6      # None = use all available dates
TARGET_RES_M = 30.0            # Shared UTM pixel spacing (m)
MIN_VALID_PCT = 80.0           # Minimum % valid unmasked pixels to keep a timestamp

# --- Strict Validation Rules ---
ALLOW_PROVISIONAL_NISAR = False    # Keep False to require strictly validated NISAR V1 products
MASK_OPERA_LAYOVER_SHADOW = True   # Use OPERA RTC-S1 *_mask.tif to remove layover/shadow/invalid px
MAX_SLOPE_DEG = 25.0               # Exclude steep slopes prone to SAR foreshortening/layover
REQUIRE_HOMOGENEOUS_3X3 = True     # Keep only pixels with uniform 3x3 label neighborhood

# --- Input Sensors & Label Sources ---
INCLUDE_OPERA_RTC_S1 = True        # Validated Sentinel-1 C-band RTC (VV, VH + mask)
INCLUDE_NISAR_GCOV = True          # Validated NISAR L-band GCOV (V1 only unless ALLOW_PROVISIONAL_NISAR=True)
INCLUDE_NISAR_GUNW = True          # Validated NISAR L-band GUNW coherence (optional extra feature)
INCLUDE_NASADEM = True             # Static terrain (elevation, slope, aspect)

INCLUDE_ESA_WORLDCOVER = True      # Static 10m validated land-cover baseline (public S3 COG)
INCLUDE_DSWX_HLS = True            # Dynamic optical water/flood validation (OPERA_L3_DSWX-HLS_V1)
HIGH_CONFIDENCE_DSWX_ONLY = True   # Keep only DSWx high-confidence water / clear non-water pixels
DSWX_MAX_TIME_DIFF_DAYS = 10       # Max |t_sar - t_dswx| days for dynamic water label matching
VALIDATED_LABELS_GEOJSON = None    # Optional path to vector polygons with integer 'class_id' column

# --- Spatial-Block Split & Sampling ---
BLOCK_SIZE_M = 3000.0              # Spatial block size in meters for leakage-free train/val/test split
SPLIT_RATIOS = (0.70, 0.15, 0.15)  # (train, val, test)
MAX_SAMPLES_PER_CLASS_PER_SPLIT = 8000
RANDOM_SEED = 42
RUN_FETCH = True

# --- Project Surface Class Schema (aligned with src/config.py) ---
IGNORE_LABEL = 255
CLASS_DEFINITIONS = {
    0: ("Bare Ground", "#64748B"),
    1: ("Water & Flood", "#0EA5E9"),
    2: ("Tree Lines & Forest", "#22C55E"),
    3: ("Wetlands", "#14B8A6"),
    4: ("Cropland", "#EAB308"),
    5: ("Urban", "#F97316"),
}

# ESA WorldCover v200 (10..100) -> Project Classes (0..5)
WORLDCOVER_TO_CLASS = {
    10: 2,   # Tree cover -> Tree Lines & Forest
    20: 0,   # Shrubland -> Bare / Sparse Ground
    30: 0,   # Grassland -> Bare / Sparse Ground
    40: 4,   # Cropland -> Cropland
    50: 5,   # Built-up -> Urban
    60: 0,   # Bare / sparse vegetation -> Bare Ground
    70: 0,   # Snow and ice -> Bare / Unclassified (or filtered)
    80: 1,   # Permanent water bodies -> Water & Flood
    90: 3,   # Herbaceous wetland -> Wetlands
    95: 3,   # Mangroves -> Wetlands
    100: 0,  # Moss and lichen -> Bare Ground
}

OUT_ROOT = TRAINING_DIR / RUN_NAME
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Configured {len(TRAINING_AOIS)} AOI(s): {list(TRAINING_AOIS.keys())}")
print(f"Strict V1 only (no provisional NISAR): {not ALLOW_PROVISIONAL_NISAR}")
print(f"Output directory: {OUT_ROOT}")


## 3. Multi-AOI Earthdata Catalog Search & Validation Audit

Queries NASA Earthdata CMR for validated (`V1`) collections only (unless `ALLOW_PROVISIONAL_NISAR=True`). Note that `OPERA_L3_DSWX-HLS_V1` is searched with `±DSWX_MAX_TIME_DIFF_DAYS` padding around `[START_DATE, END_DATE]` so edge SAR acquisitions can still find nearby optical water validation scenes.


In [ ]:
def _extract_granule_date(granule) -> str | None:
    umm = granule.get("umm", {})
    te = umm.get("TemporalExtent", {})
    rdt = te.get("RangeDateTime") or {}
    sdt = te.get("SingleDateTime")
    raw = rdt.get("BeginningDateTime") or sdt
    return str(raw)[:10] if raw else None


def search_collection(
    dataset_name: str,
    short_names: list[str],
    bbox: tuple[float, float, float, float],
    temporal: tuple[str, str] | None,
    version: str | None = None,
) -> list[dict]:
    records = []
    seen = set()
    for sn in short_names:
        kwargs = {"short_name": sn, "bounding_box": bbox}
        if temporal is not None:
            kwargs["temporal"] = temporal
        if version is not None:
            kwargs["version"] = version

        granules = earthaccess.search_data(**kwargs)
        for g in granules:
            native_id = g.get("meta", {}).get("native-id", "")
            dedup_key = native_id.replace("_PROVISIONAL_", "_")
            if dedup_key in seen:
                continue
            seen.add(dedup_key)
            raw_size = g.size() if callable(getattr(g, "size", None)) else g.get("size", 0.0)
            is_provisional = "PROVISIONAL" in sn.upper() or "_PROVISIONAL_" in native_id.upper()
            records.append(
                {
                    "dataset": dataset_name,
                    "short_name": sn,
                    "granule_id": native_id,
                    "date": _extract_granule_date(g),
                    "size_mb": float(raw_size or 0.0),
                    "validated_product": not is_provisional,
                    "granule": g,
                }
            )
    return records


def select_dates_evenly(df: pd.DataFrame, max_dates: int | None) -> pd.DataFrame:
    if df.empty or max_dates is None:
        return df
    unique_dates = sorted(d for d in df["date"].dropna().unique())
    if len(unique_dates) <= max_dates:
        return df
    idx = np.linspace(0, len(unique_dates) - 1, max_dates, dtype=int)
    chosen = {unique_dates[i] for i in idx}
    return df[df["date"].isin(chosen)].copy()


dswx_start = str((pd.Timestamp(START_DATE) - pd.Timedelta(days=DSWX_MAX_TIME_DIFF_DAYS)).date())
dswx_end = str((pd.Timestamp(END_DATE) + pd.Timedelta(days=DSWX_MAX_TIME_DIFF_DAYS)).date())

nisar_gcov_sns = ["NISAR_L2_GCOV_V1"] + (["NISAR_L2_GCOV_PROVISIONAL_V1"] if ALLOW_PROVISIONAL_NISAR else [])
nisar_gunw_sns = ["NISAR_L2_GUNW_V1"] + (["NISAR_L2_GUNW_PROVISIONAL_V1"] if ALLOW_PROVISIONAL_NISAR else [])

catalog_by_aoi: dict[str, pd.DataFrame] = {}
selected_by_aoi: dict[str, pd.DataFrame] = {}

for aoi_name, bbox in TRAINING_AOIS.items():
    plan = []
    if INCLUDE_OPERA_RTC_S1:
        plan.append(("opera_rtc_s1", ["OPERA_L2_RTC-S1_V1"], (START_DATE, END_DATE), None))
    if INCLUDE_NISAR_GCOV:
        plan.append(("nisar_gcov", nisar_gcov_sns, (START_DATE, END_DATE), None))
    if INCLUDE_NISAR_GUNW:
        plan.append(("nisar_gunw", nisar_gunw_sns, (START_DATE, END_DATE), None))
    if INCLUDE_DSWX_HLS:
        plan.append(("opera_dswx_hls", ["OPERA_L3_DSWX-HLS_V1"], (dswx_start, dswx_end), None))
    if INCLUDE_NASADEM:
        plan.append(("nasadem", ["NASADEM_HGT"], None, "001"))

    recs = []
    for ds_name, sns, temp, ver in plan:
        recs.extend(search_collection(ds_name, sns, bbox, temp, version=ver))

    cat_df = pd.DataFrame(recs)
    catalog_by_aoi[aoi_name] = cat_df
    if cat_df.empty:
        print(f"[WARN] {aoi_name}: No granules found!")
        selected_by_aoi[aoi_name] = cat_df
        continue

    sel_parts = []
    for ds_name, grp in cat_df.groupby("dataset"):
        if ds_name in ("nasadem", "opera_dswx_hls"):
            # Keep full DSWx-HLS window for temporal matching against sampled SAR dates
            sel_parts.append(grp)
        else:
            sel_parts.append(select_dates_evenly(grp, MAX_TIMESTEPS_PER_AOI))
    sel_df = pd.concat(sel_parts, ignore_index=True)
    selected_by_aoi[aoi_name] = sel_df

    audit = (
        cat_df.groupby("dataset", as_index=False)
        .agg(
            total_granules=("granule_id", "count"),
            unique_dates=("date", "nunique"),
            all_validated=("validated_product", "all"),
            first_date=("date", "min"),
            last_date=("date", "max"),
        )
        .sort_values("dataset")
    )
    print(f"\n=== Catalog Audit: {aoi_name} ===")
    display(audit)
    if INCLUDE_NISAR_GCOV and "nisar_gcov" not in set(cat_df["dataset"]):
        print(
            f"[INFO] {aoi_name}: No strictly validated NISAR_L2_GCOV_V1 granules found in {START_DATE}..{END_DATE}. "
            "OPERA_L2_RTC-S1_V1 will still be processed (or set ALLOW_PROVISIONAL_NISAR=True if needed)."
        )


## 4. Remote AOI-Only Input Fetchers with Quality & Layover/Shadow Masking

All SAR layers are streamed via `earthaccess.open()` and clipped to the padded `AOI_BBOX` before reprojection onto each AOI's UTM `TARGET_RES_M` grid:
* **OPERA `RTC-S1`:** Pairs each backscatter COG (`_VV.tif`, `_VH.tif`, `_HH.tif`, `_HV.tif`) with its granule's `*_mask.tif` and masks out any pixel with `mask != 0` (layover, shadow, invalid).
* **NISAR `GCOV`:** Reads `/science/LSAR/GCOV/grids/frequencyA` (or `frequencyB`) covariance terms plus `mask` and `numberOfLooks` when present, masking invalid or zero-look pixels.


In [ ]:
def build_target_grid(aoi_wgs84: gpd.GeoDataFrame, target_crs, res_m: float) -> xr.DataArray:
    aoi_utm = aoi_wgs84.to_crs(target_crs)
    minx, miny, maxx, maxy = aoi_utm.total_bounds
    minx = np.floor(minx / res_m) * res_m
    miny = np.floor(miny / res_m) * res_m
    maxx = np.ceil(maxx / res_m) * res_m
    maxy = np.ceil(maxy / res_m) * res_m

    xs = np.arange(minx + res_m / 2, maxx, res_m)
    ys = np.arange(maxy - res_m / 2, miny, -res_m)

    grid = xr.DataArray(
        np.zeros((len(ys), len(xs)), dtype=np.float32),
        coords={"y": ys, "x": xs},
        dims=("y", "x"),
    )
    grid = grid.rio.write_crs(target_crs).rio.set_spatial_dims(x_dim="x", y_dim="y")
    return grid


def align_to_target(da: xr.DataArray, target_grid: xr.DataArray, resampling=Resampling.bilinear) -> xr.DataArray:
    if da.rio.crs is None:
        raise ValueError("DataArray is missing CRS.")
    da = da.rio.set_spatial_dims(x_dim="x", y_dim="y")
    return da.rio.reproject_match(target_grid, resampling=resampling)


def mosaic_aligned(parts: list[xr.DataArray], mode: str = "mean") -> xr.DataArray:
    if len(parts) == 1:
        return parts[0].astype("float32")
    if mode == "first":
        out = parts[0]
        for nxt in parts[1:]:
            out = out.combine_first(nxt)
        return out.astype("float32")
    stacked = xr.concat(parts, dim="_mosaic")
    return stacked.mean(dim="_mosaic", skipna=True).astype("float32")


def to_db(da: xr.DataArray) -> xr.DataArray:
    valid = da.where(da > 0)
    return (10.0 * np.log10(valid)).astype("float32")


def _resolve_earthdata_url(granule, url: str) -> str:
    if "datapool.asf.alaska.edu" not in url:
        return url
    direct_by_name = {u.rsplit("/", 1)[-1]: u for u in granule.data_links(access="direct")}
    fname = url.rsplit("/", 1)[-1]
    s3_url = direct_by_name.get(fname, "")
    if s3_url.startswith("s3://asf-cumulus-prod-opera-products/"):
        return s3_url.replace(
            "s3://asf-cumulus-prod-opera-products/",
            "https://cumulus.asf.earthdatacloud.nasa.gov/OPERA/",
            1,
        )
    return url


def _filter_granule_urls(granules: list, suffixes: tuple[str, ...]) -> list[tuple[str, str]]:
    items = []
    for g in granules:
        dt = _extract_granule_date(g)
        for url in g.data_links(access="external"):
            if url.endswith(suffixes):
                items.append((dt, _resolve_earthdata_url(g, url)))
    return items


def _clip_remote_cog(
    file_obj,
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
    resampling=Resampling.bilinear,
    masked: bool = True,
) -> xr.DataArray | None:
    da = rioxarray.open_rasterio(file_obj, masked=masked).squeeze("band", drop=True)
    if da.rio.crs is None:
        return None
    minx, miny, maxx, maxy = aoi_wgs84.to_crs(da.rio.crs).total_bounds
    pad = TARGET_RES_M * 4.0
    try:
        clipped = da.rio.clip_box(minx - pad, miny - pad, maxx + pad, maxy + pad)
    except NoDataInBounds:
        return None
    if clipped.size == 0:
        return None
    return align_to_target(clipped, target_grid, resampling=resampling)


def _read_nisar_epsg(group: h5py.Group) -> int:
    proj = group["projection"]
    epsg_attr = proj.attrs.get("epsg_code")
    if epsg_attr is not None:
        return int(epsg_attr)
    return int(proj[()])


def _slice_h5_datasets(
    grp: h5py.Group,
    dataset_paths: list[str],
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
    resampling=Resampling.bilinear,
) -> dict[str, xr.DataArray]:
    epsg = _read_nisar_epsg(grp)
    xs = grp["xCoordinates"][:]
    ys = grp["yCoordinates"][:]

    minx, miny, maxx, maxy = aoi_wgs84.to_crs(epsg=epsg).total_bounds
    pad = TARGET_RES_M * 4.0
    xi = np.where((xs >= minx - pad) & (xs <= maxx + pad))[0]
    yi = np.where((ys >= miny - pad) & (ys <= maxy + pad))[0]
    if len(xi) == 0 or len(yi) == 0:
        return {}

    x0, x1 = int(xi[0]), int(xi[-1]) + 1
    y0, y1 = int(yi[0]), int(yi[-1]) + 1
    sub_x = xs[x0:x1].astype("float64")
    sub_y = ys[y0:y1].astype("float64")

    out = {}
    for dpath in dataset_paths:
        if dpath not in grp:
            continue
        arr = grp[dpath][y0:y1, x0:x1].astype("float32")
        da = xr.DataArray(arr, coords={"y": sub_y, "x": sub_x}, dims=("y", "x"))
        da = da.rio.write_crs(f"EPSG:{epsg}").rio.set_spatial_dims(x_dim="x", y_dim="y")
        rs = Resampling.nearest if dpath.endswith("mask") else resampling
        out[dpath] = align_to_target(da, target_grid, resampling=rs)
    return out


def fetch_nasadem_static(
    aoi_name: str,
    nasadem_df: pd.DataFrame,
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
) -> xr.Dataset | None:
    if nasadem_df.empty:
        return None
    ds_dir = RAW_DIR / f"{RUN_NAME}_{aoi_name}" / "nasadem"
    ds_dir.mkdir(parents=True, exist_ok=True)
    paths = earthaccess.download(nasadem_df["granule"].tolist(), local_path=str(ds_dir))

    hgt_files = []
    for p in paths:
        p = Path(p)
        if p.suffix.lower() == ".zip":
            with zipfile.ZipFile(p, "r") as zf:
                zf.extractall(ds_dir)
                hgt_files.extend(ds_dir / m for m in zf.namelist() if m.lower().endswith(".hgt"))
        elif p.suffix.lower() == ".hgt":
            hgt_files.append(p)

    parts = []
    for p in hgt_files:
        da = rioxarray.open_rasterio(p, masked=True).squeeze("band", drop=True)
        if da.rio.crs is None:
            da = da.rio.write_crs("EPSG:4326")
        minx, miny, maxx, maxy = aoi_wgs84.total_bounds
        da = da.rio.clip_box(minx - 0.01, miny - 0.01, maxx + 0.01, maxy + 0.01)
        parts.append(align_to_target(da, target_grid, resampling=Resampling.bilinear))

    if not parts:
        return None
    elev = mosaic_aligned(parts, mode="mean")
    dy_dx = np.gradient(elev.values, TARGET_RES_M, TARGET_RES_M)
    dz_dy, dz_dx = -dy_dx[0], dy_dx[1]
    slope_deg = np.degrees(np.arctan(np.hypot(dz_dx, dz_dy))).astype("float32")
    aspect_deg = (np.degrees(np.arctan2(dz_dy, -dz_dx)) % 360.0).astype("float32")
    aspect_rad = np.radians(aspect_deg)

    return xr.Dataset(
        {
            "elevation_m": elev,
            "slope_deg": elev.copy(data=slope_deg),
            "aspect_deg": elev.copy(data=aspect_deg),
            "aspect_sin": elev.copy(data=np.sin(aspect_rad).astype("float32")),
            "aspect_cos": elev.copy(data=np.cos(aspect_rad).astype("float32")),
        }
    )


def fetch_opera_rtc_validated(
    df: pd.DataFrame,
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
) -> xr.Dataset | None:
    if df.empty:
        return None

    pol_suffixes = ("_VV.tif", "_VH.tif", "_HH.tif", "_HV.tif")
    granule_entries = []
    urls_to_open = []
    url_roles = []  # (entry_idx, role)

    for g in df["granule"].tolist():
        dt = _extract_granule_date(g)
        pol_urls = {}
        mask_url = None
        for u in g.data_links(access="external"):
            resolved = _resolve_earthdata_url(g, u)
            if resolved.endswith(pol_suffixes):
                pol = resolved.rsplit("_", 1)[-1].replace(".tif", "")
                pol_urls[pol] = resolved
            elif MASK_OPERA_LAYOVER_SHADOW and resolved.endswith("_mask.tif"):
                mask_url = resolved
        if not pol_urls:
            continue
        idx = len(granule_entries)
        granule_entries.append({"date": dt, "pol_urls": pol_urls, "mask_url": mask_url})
        for pol, pu in pol_urls.items():
            urls_to_open.append(pu)
            url_roles.append((idx, ("pol", pol)))
        if mask_url is not None:
            urls_to_open.append(mask_url)
            url_roles.append((idx, ("mask", None)))

    if not urls_to_open:
        return None

    print(f"OPERA RTC-S1 (validated): streaming {len(urls_to_open)} COG files (incl. masks)...")
    open_files = earthaccess.open(urls_to_open, show_progress=True)

    opened_by_entry: dict[int, dict] = defaultdict(dict)
    for (entry_idx, (role, pol)), fobj in zip(url_roles, open_files):
        if role == "mask":
            opened_by_entry[entry_idx]["mask"] = fobj
        else:
            opened_by_entry[entry_idx].setdefault("pols", {})[pol] = fobj

    by_date_pol: dict[str, dict[str, list[xr.DataArray]]] = defaultdict(lambda: defaultdict(list))
    for entry_idx, entry in enumerate(granule_entries):
        dt = entry["date"]
        fmap = opened_by_entry.get(entry_idx, {})
        valid_mask = None
        if "mask" in fmap:
            m_da = _clip_remote_cog(
                fmap["mask"], aoi_wgs84, target_grid, resampling=Resampling.nearest, masked=False
            )
            if m_da is not None:
                # OPERA RTC-S1 mask: 0 = valid; non-zero = layover/shadow/invalid
                valid_mask = m_da == 0

        for pol, fobj in fmap.get("pols", {}).items():
            da = _clip_remote_cog(fobj, aoi_wgs84, target_grid, resampling=Resampling.bilinear, masked=True)
            if da is None:
                continue
            if valid_mask is not None:
                da = da.where(valid_mask)
            by_date_pol[dt][pol].append(da)

    if not by_date_pol:
        return None
    dates = sorted(by_date_pol.keys())
    all_pols = sorted({pol for pmap in by_date_pol.values() for pol in pmap})
    ds_dict = {}
    for pol in all_pols:
        per_date = [
            to_db(mosaic_aligned(by_date_pol[dt][pol], mode="mean"))
            if by_date_pol[dt].get(pol)
            else xr.full_like(target_grid, np.nan, dtype="float32")
            for dt in dates
        ]
        ds_dict[f"{pol}_dB"] = xr.concat(per_date, dim=pd.Index(pd.to_datetime(dates), name="time"))

    ds = xr.Dataset(ds_dict)
    if "VH_dB" in ds and "VV_dB" in ds:
        ds["VH_minus_VV_dB"] = (ds["VH_dB"] - ds["VV_dB"]).astype("float32")
    if "HV_dB" in ds and "HH_dB" in ds:
        ds["HH_minus_HV_dB"] = (ds["HH_dB"] - ds["HV_dB"]).astype("float32")
    return ds


def fetch_nisar_gcov_validated(
    df: pd.DataFrame,
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
) -> xr.Dataset | None:
    if df.empty:
        return None
    items = _filter_granule_urls(df["granule"].tolist(), (".h5",))
    if not items:
        return None

    urls = [u for _, u in items]
    print(f"NISAR GCOV: streaming {len(urls)} HDF5 AOI slices...")
    open_files = earthaccess.open(urls, show_progress=True)

    pol_terms = ["HHHH", "HVHV", "VVVV", "VHVH"]
    extra_layers = ["mask", "numberOfLooks"]
    by_date_term: dict[str, dict[str, list[xr.DataArray]]] = defaultdict(lambda: defaultdict(list))

    for (dt, _), fobj in zip(items, open_files):
        with h5py.File(fobj, "r") as hf:
            freq_path = "/science/LSAR/GCOV/grids/frequencyA"
            if freq_path not in hf:
                freq_path = "/science/LSAR/GCOV/grids/frequencyB"
            if freq_path not in hf:
                continue
            sliced = _slice_h5_datasets(hf[freq_path], pol_terms + extra_layers, aoi_wgs84, target_grid)
            q_mask = None
            if "mask" in sliced:
                m = sliced["mask"]
                q_mask = np.isfinite(m) & (m > 0) & (m < 255)
            if "numberOfLooks" in sliced:
                looks = sliced["numberOfLooks"]
                looks_ok = np.isfinite(looks) & (looks > 0)
                q_mask = looks_ok if q_mask is None else (q_mask & looks_ok)

            for term in pol_terms:
                if term in sliced:
                    da = sliced[term]
                    if q_mask is not None:
                        da = da.where(q_mask)
                    by_date_term[dt][term].append(da)

    if not by_date_term:
        return None
    dates = sorted(by_date_term.keys())
    present_terms = sorted({t for tmap in by_date_term.values() for t in tmap})
    ds_dict = {}
    for term in present_terms:
        per_date = [
            to_db(mosaic_aligned(by_date_term[dt][term], mode="mean"))
            if by_date_term[dt].get(term)
            else xr.full_like(target_grid, np.nan, dtype="float32")
            for dt in dates
        ]
        ds_dict[f"{term}_dB"] = xr.concat(per_date, dim=pd.Index(pd.to_datetime(dates), name="time"))

    ds = xr.Dataset(ds_dict)
    if "HHHH_dB" in ds and "HVHV_dB" in ds:
        ds["HH_minus_HV_dB"] = (ds["HHHH_dB"] - ds["HVHV_dB"]).astype("float32")
    return ds


def fetch_nisar_gunw_validated(
    df: pd.DataFrame,
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
) -> xr.Dataset | None:
    if df.empty:
        return None
    items = _filter_granule_urls(df["granule"].tolist(), (".h5",))
    if not items:
        return None

    urls = [u for _, u in items]
    print(f"NISAR GUNW: streaming {len(urls)} HDF5 AOI slices...")
    open_files = earthaccess.open(urls, show_progress=True)

    coh_paths = [f"{pol}/coherenceMagnitude" for pol in ("HH", "VV", "HV", "VH")]
    by_date_pol: dict[str, dict[str, list[xr.DataArray]]] = defaultdict(lambda: defaultdict(list))

    for (dt, _), fobj in zip(items, open_files):
        with h5py.File(fobj, "r") as hf:
            base = "/science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram"
            if base not in hf:
                continue
            sliced = _slice_h5_datasets(hf[base], coh_paths, aoi_wgs84, target_grid)
            for cpath, da in sliced.items():
                pol = cpath.split("/", 1)[0]
                by_date_pol[dt][pol].append(da.where((da >= 0.0) & (da <= 1.0)))

    if not by_date_pol:
        return None
    dates = sorted(by_date_pol.keys())
    present_pols = sorted({pol for pmap in by_date_pol.values() for pol in pmap})
    ds_dict = {}
    for pol in present_pols:
        per_date = [
            mosaic_aligned(by_date_pol[dt][pol], mode="mean")
            if by_date_pol[dt].get(pol)
            else xr.full_like(target_grid, np.nan, dtype="float32")
            for dt in dates
        ]
        ds_dict[f"coherence_{pol}"] = xr.concat(per_date, dim=pd.Index(pd.to_datetime(dates), name="time"))
    return xr.Dataset(ds_dict)


## 5. Cross-Sensor Validated Ground-Truth Label Fetchers (`y`)

Combines:
1. **ESA WorldCover 10m v200 (2021)** — Streamed AOI-only from public S3 COGs (`3° x 3°` tiles), nearest-neighbor aligned to `target_grid`, and mapped to `0..5`. With `REQUIRE_HOMOGENEOUS_3X3=True`, pixels on mixed class boundaries are masked to `255` (`IGNORE_LABEL`).
2. **OPERA L3 `DSWx-HLS` (`OPERA_L3_DSWX-HLS_V1`)** — Independent optical water classification (`_B01_WTR.tif` + `_B03_CONF.tif`). Clouds (`253`), snow/ice (`252`), ocean (`254`), and fill (`255`) are excluded. Within `±DSWX_MAX_TIME_DIFF_DAYS` of each SAR date:
   * `DSWx == 1` (open water) -> class `1` (`Water & Flood`)
   * `DSWx == 2` (partial surface water / flooded vegetation) over vegetated/cropland terrain -> class `3` (`Wetlands`)
   * `DSWx == 0` (clear non-water) over static WorldCover water (`1`) -> marks stale water pixels as `255` (`IGNORE_LABEL`) so dried-up waterbodies don't pollute training!
3. **Optional Custom Vector Polygons (`VALIDATED_LABELS_GEOJSON`)** — Overrides raster labels if supplied.


In [ ]:
def worldcover_tiles_for_bbox(bbox: tuple[float, float, float, float]) -> list[str]:
    west, south, east, north = bbox
    lat0 = int(math.floor(south / 3.0) * 3)
    lat1 = int(math.floor((north - 1e-9) / 3.0) * 3)
    lon0 = int(math.floor(west / 3.0) * 3)
    lon1 = int(math.floor((east - 1e-9) / 3.0) * 3)
    urls = []
    for lat in range(lat0, lat1 + 1, 3):
        ns = "N" if lat >= 0 else "S"
        for lon in range(lon0, lon1 + 1, 3):
            ew = "E" if lon >= 0 else "W"
            tile = f"{ns}{abs(lat):02d}{ew}{abs(lon):03d}"
            urls.append(
                f"https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/"
                f"ESA_WorldCover_10m_2021_v200_{tile}_Map.tif"
            )
    return urls


def fetch_esa_worldcover_labels(
    bbox: tuple[float, float, float, float],
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
) -> tuple[xr.DataArray, xr.DataArray] | tuple[None, None]:
    urls = worldcover_tiles_for_bbox(bbox)
    print(f"ESA WorldCover v200: streaming {len(urls)} COG tile(s) in AOI window...")
    parts = []
    for url in urls:
        try:
            da = _clip_remote_cog(
                url, aoi_wgs84, target_grid, resampling=Resampling.nearest, masked=False
            )
            if da is not None:
                parts.append(da)
        except Exception as exc:
            print(f"  [WARN] Could not load WorldCover tile {url}: {exc}")

    if not parts:
        return None, None

    raw_wc = mosaic_aligned(parts, mode="first").values.astype(np.int32)
    mapped = np.full(raw_wc.shape, IGNORE_LABEL, dtype=np.uint8)
    for wc_code, cls_id in WORLDCOVER_TO_CLASS.items():
        mapped[raw_wc == wc_code] = np.uint8(cls_id)

    # Check 3x3 label homogeneity at target grid resolution
    valid_lbl = mapped != IGNORE_LABEL
    min_nb = minimum_filter(mapped.astype(np.int16), size=3, mode="nearest")
    max_nb = maximum_filter(mapped.astype(np.int16), size=3, mode="nearest")
    is_homogeneous = valid_lbl & (min_nb == max_nb)

    lbl_da = target_grid.copy(data=mapped).astype("uint8")
    hom_da = target_grid.copy(data=is_homogeneous.astype(np.uint8)).astype("uint8")
    return lbl_da, hom_da


def fetch_dswx_hls_cube(
    df: pd.DataFrame,
    aoi_wgs84: gpd.GeoDataFrame,
    target_grid: xr.DataArray,
) -> xr.Dataset | None:
    if df.empty:
        return None

    granule_entries = []
    urls_to_open = []
    url_roles = []

    for g in df["granule"].tolist():
        dt = _extract_granule_date(g)
        wtr_url = None
        conf_url = None
        for u in g.data_links(access="external"):
            resolved = _resolve_earthdata_url(g, u)
            if resolved.endswith("_WTR.tif") and "_BWTR.tif" not in resolved:
                wtr_url = resolved
            elif HIGH_CONFIDENCE_DSWX_ONLY and resolved.endswith("_CONF.tif"):
                conf_url = resolved
        if wtr_url is None:
            continue
        idx = len(granule_entries)
        granule_entries.append({"date": dt})
        urls_to_open.append(wtr_url)
        url_roles.append((idx, "wtr"))
        if conf_url is not None:
            urls_to_open.append(conf_url)
            url_roles.append((idx, "conf"))

    if not urls_to_open:
        return None

    print(f"OPERA DSWx-HLS (validated): streaming {len(urls_to_open)} COG files...")
    open_files = earthaccess.open(urls_to_open, show_progress=True)

    opened = defaultdict(dict)
    for (idx, role), fobj in zip(url_roles, open_files):
        opened[idx][role] = fobj

    by_date_wtr: dict[str, list[xr.DataArray]] = defaultdict(list)
    for idx, entry in enumerate(granule_entries):
        fmap = opened.get(idx, {})
        if "wtr" not in fmap:
            continue
        wtr = _clip_remote_cog(
            fmap["wtr"], aoi_wgs84, target_grid, resampling=Resampling.nearest, masked=False
        )
        if wtr is None:
            continue
        # Keep only valid clear observations: 0 (not water), 1 (open water), 2 (partial surface water)
        clear = wtr.isin([0, 1, 2])
        if HIGH_CONFIDENCE_DSWX_ONLY and "conf" in fmap:
            conf = _clip_remote_cog(
                fmap["conf"], aoi_wgs84, target_grid, resampling=Resampling.nearest, masked=False
            )
            if conf is not None:
                # DSWx CONF: 1=open water high conf, 3=partial water high conf; keep clear non-water (wtr==0)
                clear = clear & ((wtr == 0) | conf.isin([1, 3]))
        wtr_clean = wtr.where(clear, np.nan).astype("float32")
        by_date_wtr[entry["date"]].append(wtr_clean)

    if not by_date_wtr:
        return None
    dates = sorted(by_date_wtr.keys())
    per_date = [mosaic_aligned(by_date_wtr[dt], mode="first") for dt in dates]
    return xr.Dataset(
        {"dswx_hls_wtr": xr.concat(per_date, dim=pd.Index(pd.to_datetime(dates), name="time"))}
    )


def rasterize_custom_geojson_labels(
    geojson_path: str | Path,
    target_grid: xr.DataArray,
) -> xr.DataArray | None:
    if geojson_path is None:
        return None
    gdf = gpd.read_file(geojson_path)
    if "class_id" not in gdf.columns:
        raise ValueError("VALIDATED_LABELS_GEOJSON must contain an integer 'class_id' column (0..5).")
    gdf = gdf.to_crs(target_grid.rio.crs)
    shapes = [(geom, int(cid)) for geom, cid in zip(gdf.geometry, gdf["class_id"]) if geom is not None]
    if not shapes:
        return None
    transform = target_grid.rio.transform()
    arr = rasterio.features.rasterize(
        shapes,
        out_shape=target_grid.shape,
        transform=transform,
        fill=IGNORE_LABEL,
        dtype=np.uint8,
    )
    return target_grid.copy(data=arr).astype("uint8")


def fuse_labels_for_timestamp(
    target_date: pd.Timestamp,
    grid_shape: tuple[int, int],
    wc_lbl: np.ndarray | None,
    wc_hom: np.ndarray | None,
    dswx_ds: xr.Dataset | None,
    custom_lbl: np.ndarray | None,
    slope_arr: np.ndarray | None,
) -> tuple[np.ndarray, str | None]:
    if wc_lbl is not None:
        lbl = wc_lbl.copy()
        if REQUIRE_HOMOGENEOUS_3X3 and wc_hom is not None:
            lbl[wc_hom == 0] = IGNORE_LABEL
    else:
        lbl = np.full(grid_shape, IGNORE_LABEL, dtype=np.uint8)

    matched_dswx_date = None
    if dswx_ds is not None and len(dswx_ds["time"]) > 0:
        dswx_times = pd.to_datetime(dswx_ds["time"].values)
        diffs = np.abs((dswx_times - target_date).days)
        best_i = int(np.argmin(diffs))
        if int(diffs[best_i]) <= DSWX_MAX_TIME_DIFF_DAYS:
            matched_dswx_date = str(dswx_times[best_i].date())
            wtr = dswx_ds["dswx_hls_wtr"].isel(time=best_i).values
            # Open water override -> class 1
            lbl[wtr == 1.0] = 1
            # Partial surface water / inundated vegetation over cropland/forest/bare -> class 3 (Wetland)
            inundated_veg = (wtr == 2.0) & np.isin(lbl, [0, 2, 3, 4])
            lbl[inundated_veg] = 3
            # Clear non-water over static permanent water -> mark uncertain (255) to avoid stale labels
            stale_water = (wtr == 0.0) & (lbl == 1)
            lbl[stale_water] = IGNORE_LABEL

    # Custom validated polygons take highest precedence
    if custom_lbl is not None:
        has_custom = custom_lbl != IGNORE_LABEL
        lbl[has_custom] = custom_lbl[has_custom]

    # Mask steep slopes prone to radar layover/foreshortening distortion
    if slope_arr is not None:
        bad_slope = (~np.isfinite(slope_arr)) | (slope_arr > MAX_SLOPE_DEG)
        lbl[bad_slope] = IGNORE_LABEL

    return lbl, matched_dswx_date


## 6. Execute Acquisition, Build Spatially-Blocked Features & Export Training Sets

For each AOI:
1. Streams & aligns validated cubes (`nasadem_static`, `opera_rtc_s1`, `nisar_gcov`, `nisar_gunw`, `esa_worldcover`, `opera_dswx_hls`) to the AOI UTM grid and saves compressed NetCDF-4 files under `data/training/<RUN_NAME>/<aoi>/`.
2. Builds `3x3` texture-augmented SAR + terrain feature vectors **only where the full `3x3` neighborhood is free of masked/NaN pixels** (`binary_dilation` guard).
3. Assigns deterministic `BLOCK_SIZE_M` spatial blocks to `train` / `val` / `test` (`SPLIT_RATIOS`) across all timestamps to prevent spatial leakage.
4. Exports class-balanced `.npz` & `.parquet` datasets per sensor (`opera_rtc_s1` and/or `nisar_gcov`) plus `dataset_metadata.json` and `validation_manifest.csv`.


In [ ]:
def assign_spatial_block_splits(
    aoi_name: str,
    xs: np.ndarray,
    ys: np.ndarray,
    block_size_m: float,
    split_ratios: tuple[float, float, float],
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    """Return (split_grid [0=train,1=val,2=test], block_id_grid) for a 2D UTM grid."""
    bx = np.floor(xs / block_size_m).astype(np.int64)[None, :]
    by = np.floor(ys / block_size_m).astype(np.int64)[:, None]
    bx_2d = np.broadcast_to(bx, (len(ys), len(xs)))
    by_2d = np.broadcast_to(by, (len(ys), len(xs)))

    t_thr = split_ratios[0]
    v_thr = split_ratios[0] + split_ratios[1]

    split_grid = np.empty((len(ys), len(xs)), dtype=np.uint8)
    block_id_grid = np.empty((len(ys), len(xs)), dtype=object)

    for ux in np.unique(bx):
        x_mask = bx_2d[0, :] == ux
        for uy in np.unique(by):
            y_mask = by_2d[:, 0] == uy
            digest = hashlib.sha256(f"{seed}:{aoi_name}:{int(ux)}:{int(uy)}".encode("utf-8")).digest()
            u01 = int.from_bytes(digest[:8], "big") / float(1 << 64)
            sp = 0 if u01 < t_thr else (1 if u01 < v_thr else 2)
            split_grid[np.ix_(y_mask, x_mask)] = sp
            block_id_grid[np.ix_(y_mask, x_mask)] = f"{aoi_name}_{int(ux)}_{int(uy)}"
    return split_grid, block_id_grid


def extract_3x3_texture_features(
    band_arrays: dict[str, np.ndarray],
    static_arrays: dict[str, np.ndarray],
) -> tuple[np.ndarray, list[str], np.ndarray]:
    """
    Build feature array (H, W, F) with raw dB, 3x3 mean, 3x3 std, and static terrain.
    Returns (features_hwf, feature_names, valid_3x3_mask_hw).
    """
    valid_2d = np.ones(next(iter(band_arrays.values())).shape, dtype=bool)
    for arr in band_arrays.values():
        valid_2d &= np.isfinite(arr)
    for arr in static_arrays.values():
        valid_2d &= np.isfinite(arr)

    # Ensure 3x3 neighborhood has zero NaNs/masked pixels before computing uniform_filter
    valid_3x3 = ~binary_dilation(~valid_2d, structure=np.ones((3, 3), dtype=bool))

    feat_channels = []
    feat_names = []
    for name, arr in band_arrays.items():
        clean = np.where(valid_2d, arr, 0.0).astype(np.float32)
        feat_channels.append(clean)
        feat_names.append(name)
        if name.endswith("_dB") and "minus" not in name:
            mu = uniform_filter(clean, size=3, mode="nearest")
            sq = uniform_filter(clean * clean, size=3, mode="nearest")
            sd = np.sqrt(np.maximum(sq - mu * mu, 0.0)).astype(np.float32)
            feat_channels.extend([mu.astype(np.float32), sd])
            feat_names.extend([f"{name}_mean3", f"{name}_std3"])

    for name, arr in static_arrays.items():
        clean = np.where(valid_2d, arr, 0.0).astype(np.float32)
        feat_channels.append(clean)
        feat_names.append(name)

    stacked = np.stack(feat_channels, axis=-1).astype(np.float32)
    return stacked, feat_names, valid_3x3


def collect_sensor_samples(
    aoi_name: str,
    sensor_name: str,
    sensor_ds: xr.Dataset,
    static_ds: xr.Dataset | None,
    gunw_ds: xr.Dataset | None,
    wc_lbl: np.ndarray | None,
    wc_hom: np.ndarray | None,
    dswx_ds: xr.Dataset | None,
    custom_lbl: np.ndarray | None,
    split_grid: np.ndarray,
    block_id_grid: np.ndarray,
) -> tuple[list[pd.DataFrame], xr.DataArray, list[dict]]:
    static_arrays = {}
    slope_arr = None
    if static_ds is not None:
        for k in ("elevation_m", "slope_deg", "aspect_sin", "aspect_cos"):
            if k in static_ds:
                static_arrays[k] = static_ds[k].values.astype(np.float32)
        slope_arr = static_arrays.get("slope_deg")

    ys_2d = np.broadcast_to(sensor_ds["y"].values[:, None], split_grid.shape)
    xs_2d = np.broadcast_to(sensor_ds["x"].values[None, :], split_grid.shape)

    frames = []
    label_slices = []
    manifest_rows = []
    feature_names = None

    for t_val in sensor_ds["time"].values:
        t_ts = pd.Timestamp(t_val)
        date_str = str(t_ts.date())
        band_arrays = {
            var: sensor_ds[var].sel(time=t_val).values.astype(np.float32)
            for var in sensor_ds.data_vars
        }

        # Attach closest NISAR GUNW coherence if available within 12 days
        if gunw_ds is not None and "coherence_HH" in gunw_ds and len(gunw_ds["time"]) > 0:
            g_times = pd.to_datetime(gunw_ds["time"].values)
            diffs = np.abs((g_times - t_ts).days)
            bi = int(np.argmin(diffs))
            if int(diffs[bi]) <= 12:
                band_arrays["coherence_HH"] = gunw_ds["coherence_HH"].isel(time=bi).values.astype(np.float32)

        feat_hwf, fnames, valid_3x3 = extract_3x3_texture_features(band_arrays, static_arrays)
        if feature_names is None:
            feature_names = fnames

        lbl_2d, matched_dswx = fuse_labels_for_timestamp(
            t_ts, split_grid.shape, wc_lbl, wc_hom, dswx_ds, custom_lbl, slope_arr
        )
        sample_mask = valid_3x3 & (lbl_2d != IGNORE_LABEL)
        valid_pct = float(valid_3x3.mean() * 100.0)
        labeled_px = int(sample_mask.sum())

        label_da = xr.DataArray(lbl_2d, coords={"y": sensor_ds["y"], "x": sensor_ds["x"]}, dims=("y", "x"))
        label_slices.append(label_da)

        manifest_rows.append(
            {
                "aoi": aoi_name,
                "sensor": sensor_name,
                "date": date_str,
                "dswx_matched_date": matched_dswx or "none",
                "valid_3x3_pct": round(valid_pct, 2),
                "labeled_pixels": labeled_px,
            }
        )

        if valid_pct < MIN_VALID_PCT or labeled_px == 0:
            continue

        df_t = pd.DataFrame(feat_hwf[sample_mask], columns=fnames)
        df_t["label"] = lbl_2d[sample_mask].astype(np.int64)
        df_t["split"] = split_grid[sample_mask].astype(np.uint8)
        df_t["block_id"] = block_id_grid[sample_mask]
        df_t["aoi"] = aoi_name
        df_t["date"] = date_str
        df_t["x_utm"] = xs_2d[sample_mask]
        df_t["y_utm"] = ys_2d[sample_mask]
        frames.append(df_t)

    labels_cube = xr.concat(
        label_slices, dim=pd.Index(pd.to_datetime(sensor_ds["time"].values), name="time")
    )
    return frames, labels_cube, manifest_rows


def stratified_sample_and_export(
    sensor_name: str,
    df_all: pd.DataFrame,
    out_dir: Path,
) -> dict:
    rng = np.random.default_rng(RANDOM_SEED)
    meta_cols = {"label", "split", "block_id", "aoi", "date", "x_utm", "y_utm"}
    feature_cols = [c for c in df_all.columns if c not in meta_cols]

    sampled_parts = []
    for sp_id in (0, 1, 2):
        df_sp = df_all[df_all["split"] == sp_id]
        for cls_id, grp in df_sp.groupby("label"):
            if len(grp) > MAX_SAMPLES_PER_CLASS_PER_SPLIT:
                idx = rng.choice(grp.index.to_numpy(), size=MAX_SAMPLES_PER_CLASS_PER_SPLIT, replace=False)
                sampled_parts.append(grp.loc[idx])
            else:
                sampled_parts.append(grp)

    balanced_df = pd.concat(sampled_parts, ignore_index=True)
    parquet_path = out_dir / f"training_samples_{sensor_name}.parquet"
    balanced_df.to_parquet(parquet_path, index=False)

    split_arrays = {}
    split_names = {0: "train", 1: "val", 2: "test"}
    counts_summary = {}
    for sp_id, sp_name in split_names.items():
        sub = balanced_df[balanced_df["split"] == sp_id]
        split_arrays[f"X_{sp_name}"] = sub[feature_cols].to_numpy(dtype=np.float32)
        split_arrays[f"y_{sp_name}"] = sub["label"].to_numpy(dtype=np.int64)
        counts_summary[sp_name] = {int(k): int(v) for k, v in sub["label"].value_counts().sort_index().items()}

    npz_path = out_dir / f"training_samples_{sensor_name}.npz"
    np.savez_compressed(
        npz_path,
        **split_arrays,
        feature_names=np.array(feature_cols),
        class_ids=np.array(sorted(CLASS_DEFINITIONS.keys()), dtype=np.int64),
    )
    print(f"[EXPORT] {sensor_name}: {len(balanced_df):,} balanced samples -> {npz_path.name} & {parquet_path.name}")
    return {"feature_cols": feature_cols, "counts": counts_summary, "total_rows": len(balanced_df)}


sensor_frames: dict[str, list[pd.DataFrame]] = defaultdict(list)
all_manifest_rows: list[dict] = []
qa_artifacts: dict[str, dict] = {}

for aoi_name, bbox in TRAINING_AOIS.items():
    aoi_gdf = gpd.GeoDataFrame(geometry=[box(*bbox)], crs="EPSG:4326")
    utm_crs = aoi_gdf.estimate_utm_crs()
    target_grid = build_target_grid(aoi_gdf, utm_crs, TARGET_RES_M)
    aoi_out = OUT_ROOT / aoi_name
    aoi_out.mkdir(parents=True, exist_ok=True)
    sel_df = selected_by_aoi.get(aoi_name, pd.DataFrame())

    print(f"\n================ Processing AOI: {aoi_name} ({utm_crs}) ================")
    if RUN_FETCH:
        nasadem_ds = (
            fetch_nasadem_static(aoi_name, sel_df[sel_df["dataset"] == "nasadem"], aoi_gdf, target_grid)
            if INCLUDE_NASADEM else None
        )
        rtc_ds = (
            fetch_opera_rtc_validated(sel_df[sel_df["dataset"] == "opera_rtc_s1"], aoi_gdf, target_grid)
            if INCLUDE_OPERA_RTC_S1 else None
        )
        gcov_ds = (
            fetch_nisar_gcov_validated(sel_df[sel_df["dataset"] == "nisar_gcov"], aoi_gdf, target_grid)
            if INCLUDE_NISAR_GCOV else None
        )
        gunw_ds = (
            fetch_nisar_gunw_validated(sel_df[sel_df["dataset"] == "nisar_gunw"], aoi_gdf, target_grid)
            if INCLUDE_NISAR_GUNW else None
        )
        wc_lbl_da, wc_hom_da = (
            fetch_esa_worldcover_labels(bbox, aoi_gdf, target_grid)
            if INCLUDE_ESA_WORLDCOVER else (None, None)
        )
        dswx_ds = (
            fetch_dswx_hls_cube(sel_df[sel_df["dataset"] == "opera_dswx_hls"], aoi_gdf, target_grid)
            if INCLUDE_DSWX_HLS else None
        )
        custom_lbl_da = rasterize_custom_geojson_labels(VALIDATED_LABELS_GEOJSON, target_grid)

        for cname, cds in (
            ("nasadem_static", nasadem_ds),
            ("opera_rtc_s1", rtc_ds),
            ("nisar_gcov", gcov_ds),
            ("nisar_gunw", gunw_ds),
            ("opera_dswx_hls", dswx_ds),
        ):
            if cds is not None:
                enc = {v: {"zlib": True, "complevel": 4} for v in cds.data_vars}
                cds.to_netcdf(aoi_out / f"{cname}.nc", engine="h5netcdf", encoding=enc)
        if wc_lbl_da is not None:
            wc_ds = xr.Dataset({"worldcover_class": wc_lbl_da, "is_homogeneous_3x3": wc_hom_da})
            wc_ds.to_netcdf(aoi_out / "worldcover_static.nc", engine="h5netcdf")
    else:
        def _load_opt(fname):
            p = aoi_out / fname
            if not p.exists():
                return None
            with xr.open_dataset(p, engine="h5netcdf") as ds_in:
                return ds_in.load()

        nasadem_ds = _load_opt("nasadem_static.nc")
        rtc_ds = _load_opt("opera_rtc_s1.nc")
        gcov_ds = _load_opt("nisar_gcov.nc")
        gunw_ds = _load_opt("nisar_gunw.nc")
        dswx_ds = _load_opt("opera_dswx_hls.nc")
        wc_ds = _load_opt("worldcover_static.nc")
        wc_lbl_da = wc_ds["worldcover_class"] if wc_ds is not None else None
        wc_hom_da = wc_ds["is_homogeneous_3x3"] if wc_ds is not None else None
        custom_lbl_da = rasterize_custom_geojson_labels(VALIDATED_LABELS_GEOJSON, target_grid)

    wc_lbl_arr = wc_lbl_da.values if wc_lbl_da is not None else None
    wc_hom_arr = wc_hom_da.values if wc_hom_da is not None else None
    custom_arr = custom_lbl_da.values if custom_lbl_da is not None else None

    split_grid, block_id_grid = assign_spatial_block_splits(
        aoi_name, target_grid["x"].values, target_grid["y"].values, BLOCK_SIZE_M, SPLIT_RATIOS, RANDOM_SEED
    )

    for s_name, s_ds in (("opera_rtc_s1", rtc_ds), ("nisar_gcov", gcov_ds)):
        if s_ds is None:
            continue
        frames, lbl_cube, mrows = collect_sensor_samples(
            aoi_name, s_name, s_ds, nasadem_ds, gunw_ds,
            wc_lbl_arr, wc_hom_arr, dswx_ds, custom_arr, split_grid, block_id_grid,
        )
        xr.Dataset({"label": lbl_cube}).to_netcdf(
            aoi_out / f"labels_{s_name}.nc",
            engine="h5netcdf",
            encoding={"label": {"zlib": True, "complevel": 4}},
        )
        sensor_frames[s_name].extend(frames)
        all_manifest_rows.extend(mrows)
        qa_artifacts[f"{aoi_name}:{s_name}"] = {
            "sensor_ds": s_ds,
            "labels_cube": lbl_cube,
            "split_grid": split_grid,
            "nasadem_ds": nasadem_ds,
        }

manifest_df = pd.DataFrame(all_manifest_rows)
if not manifest_df.empty:
    manifest_df.to_csv(OUT_ROOT / "validation_manifest.csv", index=False)
    display(manifest_df)

export_summary = {}
for s_name, dfs in sensor_frames.items():
    if dfs:
        combined = pd.concat(dfs, ignore_index=True)
        export_summary[s_name] = stratified_sample_and_export(s_name, combined, OUT_ROOT)

metadata = {
    "run_name": RUN_NAME,
    "allow_provisional_nisar": ALLOW_PROVISIONAL_NISAR,
    "mask_opera_layover_shadow": MASK_OPERA_LAYOVER_SHADOW,
    "max_slope_deg": MAX_SLOPE_DEG,
    "require_homogeneous_3x3": REQUIRE_HOMOGENEOUS_3X3,
    "dswx_max_time_diff_days": DSWX_MAX_TIME_DIFF_DAYS,
    "block_size_m": BLOCK_SIZE_M,
    "split_ratios": list(SPLIT_RATIOS),
    "classes": {str(k): v[0] for k, v in CLASS_DEFINITIONS.items()},
    "exports": export_summary,
}
(OUT_ROOT / "dataset_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved metadata ->", OUT_ROOT / "dataset_metadata.json")


## 7. Validation QA & Feature Separation Diagnostics

In [ ]:
if not qa_artifacts:
    print("No sensor datasets exported yet; nothing to plot.")
else:
    first_key = next(iter(qa_artifacts))
    qa = qa_artifacts[first_key]
    s_ds = qa["sensor_ds"]
    lbl_cube = qa["labels_cube"]
    split_grid = qa["split_grid"]

    # Pick timestamp with most labeled pixels
    lbl_counts = [(lbl_cube.isel(time=i).values != IGNORE_LABEL).sum() for i in range(len(lbl_cube["time"]))]
    best_idx = int(np.argmax(lbl_counts))
    best_date = str(pd.Timestamp(lbl_cube["time"].values[best_idx]).date())
    lbl_2d = lbl_cube.isel(time=best_idx).values

    first_var = next(iter(s_ds.data_vars))
    sar_2d = s_ds[first_var].isel(time=best_idx).values

    fig, axes = plt.subplots(2, 2, figsize=(14, 11))

    # 1. SAR Backscatter (masked)
    axes[0, 0].imshow(sar_2d, cmap="gray", vmin=-25, vmax=0)
    axes[0, 0].set_title(f"{first_key} — {first_var} ({best_date})")

    # 2. Validated Fused Labels
    cmap_colors = [CLASS_DEFINITIONS[i][1] for i in range(6)] + ["#1E293B"]
    cmap = mcolors.ListedColormap(cmap_colors)
    lbl_plot = np.where(lbl_2d == IGNORE_LABEL, 6, lbl_2d)
    im = axes[0, 1].imshow(lbl_plot, cmap=cmap, vmin=0, vmax=6, interpolation="nearest")
    axes[0, 1].set_title("Validated Labels (Dark = Masked/Ignored)")
    cbar = plt.colorbar(im, ax=axes[0, 1], ticks=list(range(7)))
    cbar.ax.set_yticklabels([CLASS_DEFINITIONS[i][0] for i in range(6)] + ["Ignored (255)"])

    # 3. Spatial Block Split
    split_cmap = mcolors.ListedColormap(["#3B82F6", "#F59E0B", "#EF4444"])
    im_sp = axes[1, 0].imshow(split_grid, cmap=split_cmap, vmin=0, vmax=2, interpolation="nearest")
    axes[1, 0].set_title(f"Spatial Block Split ({BLOCK_SIZE_M:.0f}m blocks: Train/Val/Test)")
    cbar_sp = plt.colorbar(im_sp, ax=axes[1, 0], ticks=[0, 1, 2])
    cbar_sp.ax.set_yticklabels(["Train", "Val", "Test"])

    # 4. Feature Separation Scatter (co-pol vs cross-pol)
    sensor_only = first_key.split(":", 1)[1]
    pq_path = OUT_ROOT / f"training_samples_{sensor_only}.parquet"
    if pq_path.exists():
        sample_df = pd.read_parquet(pq_path)
        plot_sub = sample_df.sample(min(len(sample_df), 4000), random_state=RANDOM_SEED)
        x_col = "VV_dB" if "VV_dB" in plot_sub else ("HHHH_dB" if "HHHH_dB" in plot_sub else first_var)
        y_col = "VH_dB" if "VH_dB" in plot_sub else ("HVHV_dB" if "HVHV_dB" in plot_sub else first_var)
        for cid, (cname, c_hex) in CLASS_DEFINITIONS.items():
            sub_c = plot_sub[plot_sub["label"] == cid]
            if not sub_c.empty:
                axes[1, 1].scatter(sub_c[x_col], sub_c[y_col], s=6, alpha=0.45, color=c_hex, label=cname)
        axes[1, 1].set_xlabel(x_col)
        axes[1, 1].set_ylabel(y_col)
        axes[1, 1].set_title("Validated Class Backscatter Separation")
        axes[1, 1].legend(markerscale=2, fontsize=8)
    else:
        axes[1, 1].axis("off")

    for ax in axes.ravel()[:3]:
        ax.set_xticks([])
        ax.set_yticks([])

    plt.tight_layout()
    plt.show()
